# CSE 291 / DSC 291 PA3 — Speculative Decoding

In this notebook you will implement and benchmark a single-sequence (batch=1) speculative decoder.

Recap of the algorithm:

1. A small **draft** model proposes `k` tokens autoregressively starting from the current context.
2. The large **target** model verifies the proposal in **one** forward pass (a single batched pass over the `L + k` length sequence).
3. Tokens are accepted greedily up to the first mismatch with the target's argmax. After the first mismatch, the target's own next token is appended and the loop restarts.

Default model pair (public weights, runs on any GPU with >=4 GB VRAM):

- target: `EleutherAI/pythia-1.4b-deduped`
- draft:  `EleutherAI/pythia-160m-deduped`

If you don't have GPU access, the same code paths run on CPU but you won't see a meaningful speedup.

## Setup

In [6]:
import os
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Speculative Decoder

In [7]:
class SpeculativeDecoder:
    def __init__(self, target_model_name: str, draft_model_name: str, device: str = "cuda"):
        """Initialize the speculative decoder with a target and a draft model."""
        self.device = device
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)

        assert self.target_tokenizer.get_vocab() == self.draft_tokenizer.get_vocab(), (
            "Target and draft must share a vocabulary"
        )

    def initialize_target_model(self, model_name: str):
        """Load the larger target model with caching enabled."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        # 1. Make sure tokenizer has a pad token (set to eos if missing).
        # 2. Load the model in an inference-friendly dtype (fp16 / bf16) on self.device.
        # 3. Put the model in eval() mode and enable KV caching.
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        dtype = torch.float16 if self.device.startswith("cuda") else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype = dtype,
            low_cpu_mem_usage=True
        ).to(self.device)

        model.eval()
        model.config.use_cache = True

        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """Load the smaller draft model."""
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        # TODO (3.1): same as the target initializer, but for the draft.
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        dtype = torch.float16 if self.device.startswith("cuda") else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype = dtype,
            low_cpu_mem_usage=True
        ).to(self.device)

        model.eval()
        model.config.use_cache = True

        return model, tokenizer

    def generate_draft_tokens(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                             num_speculative_tokens: int = 10) -> torch.Tensor:
        """
        Generate `num_speculative_tokens` draft tokens with the draft model.

        Args:
            input_ids: Input token IDs (tensor of shape [1, seq_len]).
            attention_mask: Corresponding attention mask.
            num_speculative_tokens: Number of tokens to speculate.

        Returns:
            Tensor of shape [1, num_speculative_tokens] containing the draft tokens.
        """
        # 1. Use the draft model to generate tokens
        # 2. Extract only the new tokens (not including the input)
        # 3. Return the newly generated tokens
        with torch.inference_mode():
            output_ids = self.draft_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=num_speculative_tokens,
                do_sample=False,
                pad_token_id=self.draft_tokenizer.pad_token_id,
                eos_token_id=self.draft_tokenizer.eos_token_id,
                use_cache=getattr(self, "draft_use_cache", True)
            )

        return output_ids[:, input_ids.shape[1]:]

    def verify_tokens_vectorized(self, input_ids: torch.Tensor, draft_tokens: torch.Tensor,
                               attention_mask: torch.Tensor) -> Tuple[List[int], int]:
        """
        Vectorized verification: verify all draft tokens in one forward pass using the target model.

        Args:
            input_ids: The current input token IDs (shape [1, L]).
            draft_tokens: Draft tokens from the draft model (shape [1, k]).
            attention_mask: The current attention mask for input_ids.

        Returns:
            accepted_tokens: List of accepted token IDs.
            accepted_position: Index of the first rejected token (if all accepted, equals draft_tokens.shape[1]).
        """
        # 1. Run target model on input_ids concatenated with draft_tokens
        # 2. Extract the logits for positions where draft tokens would be predicted
        # 3. Compare target model predictions with draft tokens
        # 4. Determine how many consecutive tokens were accepted before first mismatch
        k = draft_tokens.shape[1]
        if k == 0:
            self._last_target_token = None
            return [], 0

        full_input_ids = torch.cat([input_ids, draft_tokens], dim=1)
        draft_attention = torch.ones_like(draft_tokens, dtype=attention_mask.dtype)
        full_attention_mask = torch.cat([attention_mask, draft_attention], dim=1)

        with torch.inference_mode():
            outputs = self.target_model(
                input_ids=full_input_ids,
                attention_mask=full_attention_mask,
                use_cache=True,
            )

        start = input_ids.shape[1] - 1
        end = start + k + 1
        target_predictions = torch.argmax(outputs.logits[:, start:end, :], dim=-1)

        draft_list = draft_tokens[0].tolist()
        target_list = target_predictions[0].tolist()

        accepted_tokens = []
        for pos, draft_token in enumerate(draft_list):
            if int(draft_token) == int(target_list[pos]):
                accepted_tokens.append(int(draft_token))
            else:
                break

        accepted_position = len(accepted_tokens)
        self._last_target_token = int(target_list[accepted_position])

        return accepted_tokens, accepted_position


    def speculative_decode(self, prompt: str, max_tokens: int = 100,
                          num_speculative_tokens: int = 8) -> str:
        """
        Main speculative decoding algorithm with vectorized verification.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate (excluding prompt).
            num_speculative_tokens: Number of tokens to speculate per iteration.

        Returns:
            Generated text.
        """
        # Tokenize prompt
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length = input_ids.shape[1]

        # Initialize counters for performance tracking
        total_tokens_generated = prompt_length
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        start_time = time.time()

        # 1. Generate draft tokens using the draft model
        # 2. Verify draft tokens using the target model
        # 3. Accept verified tokens and append to the sequence
        # 4. For rejected tokens or if all tokens are accepted, generate a new token with the target model
        # 5. Stop when max_tokens is reached or an EOS token is generated
        eos_token = self.target_tokenizer.eos_token_id

        while total_tokens_generated - prompt_length < max_tokens:
            remaining = max_tokens - (total_tokens_generated - prompt_length)
            k = min(num_speculative_tokens, remaining)

            draft_tokens = self.generate_draft_tokens(input_ids, attention_mask, k)
            if draft_tokens.shape[1] == 0:
                break
            
            total_draft_tokens_proposed += draft_tokens.shape[1]

            accepted_tokens, _ = self.verify_tokens_vectorized(
                input_ids,
                draft_tokens,
                attention_mask
            )

            total_draft_tokens_accepted += len(accepted_tokens)

            new_tokens = list(accepted_tokens)
            if len(new_tokens) < remaining and self._last_target_token is not None:
                new_tokens.append(self._last_target_token)
            
            new_tokens = new_tokens[:remaining]
            if not new_tokens:
                break
            
            should_stop = False
            if eos_token in new_tokens:
                eos_pos = new_tokens.index(eos_token) + 1
                new_tokens = new_tokens[:eos_pos]
                should_stop = True

            new_token_ids = torch.tensor(
                [new_tokens],
                dtype=input_ids.dtype,
                device=self.device,
            )
            input_ids = torch.cat([input_ids, new_token_ids], dim=1)
            attention_mask = torch.cat(
                [attention_mask, torch.ones_like(new_token_ids, dtype=attention_mask.dtype)],
                dim=1,
            )

            total_tokens_generated += len(new_tokens)

            if should_stop:
                break


        # Calculate performance metrics
        elapsed_time = time.time() - start_time
        acceptance_rate = total_draft_tokens_accepted / total_draft_tokens_proposed if total_draft_tokens_proposed > 0 else 0

        print(f"Generated {total_tokens_generated - prompt_length} tokens in {elapsed_time:.2f} seconds")
        print(f"Tokens per second: {(total_tokens_generated - prompt_length) / elapsed_time:.2f}")
        print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        generated_tokens = total_tokens_generated - prompt_length
        tokens_per_second = generated_tokens / elapsed_time if elapsed_time > 0 else 0
        acceptance_rate = total_draft_tokens_accepted / total_draft_tokens_proposed if total_draft_tokens_proposed > 0 else 0

        self.last_stats = {
            "elapsed_time": elapsed_time,
            "tokens_per_second": tokens_per_second,
            "acceptance_rate": acceptance_rate,
            "draft_tokens_proposed": total_draft_tokens_proposed,
            "draft_tokens_accepted": total_draft_tokens_accepted,
            "generated_tokens": generated_tokens,
        }

        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_runs: int = 3,
        compare_baseline: bool = True,
        num_speculative_tokens: int = 8,
    ) -> Dict:
        results = {
            "speculative": {
                "times": [],
                "tokens_per_second": [],
                "acceptance_rates": [],
            },
            "baseline": {
                "times": [],
                "tokens_per_second": [],
            } if compare_baseline else None,
        }

        for _ in range(num_runs):
            _ = self.speculative_decode(
                prompt,
                max_tokens=max_tokens,
                num_speculative_tokens=num_speculative_tokens,
            )

            stats = self.last_stats
            results["speculative"]["times"].append(stats["elapsed_time"])
            results["speculative"]["tokens_per_second"].append(stats["tokens_per_second"])
            results["speculative"]["acceptance_rates"].append(stats["acceptance_rate"])

        if compare_baseline:
            for _ in range(num_runs):
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)

                if self.device.startswith("cuda"):
                    torch.cuda.synchronize()
                t0 = time.time()

                with torch.inference_mode():
                    output_ids = self.target_model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_tokens,
                        do_sample=False,
                        pad_token_id=self.target_tokenizer.pad_token_id,
                        eos_token_id=self.target_tokenizer.eos_token_id,
                        use_cache=True
                    )

                if self.device.startswith("cuda"):
                    torch.cuda.synchronize()
                elapsed = time.time() - t0

                output_tokens = output_ids.shape[1] - input_ids.shape[1]
                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(output_tokens / elapsed)

        spec = results["speculative"]
        spec["avg_time"] = sum(spec["times"]) / num_runs
        spec["avg_tokens_per_second"] = sum(spec["tokens_per_second"]) / num_runs
        spec["avg_acceptance_rate"] = sum(spec["acceptance_rates"]) / num_runs

        if compare_baseline:
            base = results["baseline"]
            base["avg_time"] = sum(base["times"]) / num_runs
            base["avg_tokens_per_second"] = sum(base["tokens_per_second"]) / num_runs

            results["speedup"] = base["avg_time"] / spec["avg_time"]
            results["latency_reduction"] = (
                1 - spec["avg_time"] / base["avg_time"]
            ) * 100

        return results

    # def benchmark(
    #     self,
    #     prompt: str,
    #     max_tokens: int = 100,
    #     num_runs: int = 3,
    #     compare_baseline: bool = True,
    # ) -> Dict:
    #     results = {
    #         "speculative": {"times": [], "tokens_per_second": []},
    #         "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None,
    #     }

    #     for _ in range(num_runs):
    #         t0 = time.time()
    #         output = self.speculative_decode(prompt, max_tokens=max_tokens)
    #         elapsed = time.time() - t0
    #         prompt_len = len(self.target_tokenizer(prompt)["input_ids"])
    #         output_tokens = len(self.target_tokenizer.encode(output)) - prompt_len
    #         results["speculative"]["times"].append(elapsed)
    #         results["speculative"]["tokens_per_second"].append(output_tokens / elapsed)

    #     if compare_baseline:
    #         for _ in range(num_runs):
    #             inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
    #             input_ids = inputs["input_ids"].to(self.device)
    #             attention_mask = inputs["attention_mask"].to(self.device)
    #             t0 = time.time()
    #             with torch.no_grad():
    #                 output_ids = self.target_model.generate(
    #                     input_ids,
    #                     attention_mask=attention_mask,
    #                     max_length=input_ids.shape[1] + max_tokens,
    #                     do_sample=False,
    #                     pad_token_id=self.target_tokenizer.pad_token_id,
    #                 )
    #             elapsed = time.time() - t0
    #             output_tokens = output_ids.shape[1] - input_ids.shape[1]
    #             results["baseline"]["times"].append(elapsed)
    #             results["baseline"]["tokens_per_second"].append(output_tokens / elapsed)

    #     for method in results:
    #         if results[method] is not None:
    #             results[method]["avg_time"] = sum(results[method]["times"]) / num_runs
    #             results[method]["avg_tokens_per_second"] = (
    #                 sum(results[method]["tokens_per_second"]) / num_runs
    #             )
    #     if compare_baseline:
    #         results["speedup"] = (
    #             results["baseline"]["avg_time"] / results["speculative"]["avg_time"]
    #         )
    #         results["latency_reduction"] = (
    #             1 - results["speculative"]["avg_time"] / results["baseline"]["avg_time"]
    #         ) * 100
    #     return results

## Test

In [8]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"
draft_model_name = "EleutherAI/pythia-160m-deduped"

decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

test_prompts = [
    "The future of artificial intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}: {prompt}")
    results = decoder.benchmark(prompt=prompt, max_tokens=100, num_runs=3, compare_baseline=True)
    print(f"  Speculative: {results['speculative']['avg_time']:.2f}s, "
          f"{results['speculative']['avg_tokens_per_second']:.2f} tok/s")
    if results['baseline'] is not None:
        print(f"  Baseline:    {results['baseline']['avg_time']:.2f}s, "
              f"{results['baseline']['avg_tokens_per_second']:.2f} tok/s")
        print(f"  Speedup: {results['speedup']:.2f}x  |  Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: EleutherAI/pythia-1.4b-deduped


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 421.57it/s] 


Loading draft model: EleutherAI/pythia-160m-deduped


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1735.39it/s]



Benchmarking Prompt 1: The future of artificial intelligence is
Generated 100 tokens in 0.61 seconds
Tokens per second: 164.83
Draft token acceptance rate: 91.67%
Generated 100 tokens in 0.59 seconds
Tokens per second: 170.23
Draft token acceptance rate: 91.67%
Generated 100 tokens in 0.61 seconds
Tokens per second: 163.13
Draft token acceptance rate: 91.67%
  Speculative: 0.60s, 166.06 tok/s
  Baseline:    0.84s, 118.48 tok/s
  Speedup: 1.40x  |  Latency reduction: 28.66%

Benchmarking Prompt 2: Write a short story about a robot learning to feel emotions:
Generated 100 tokens in 0.54 seconds
Tokens per second: 186.03
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.55 seconds
Tokens per second: 182.80
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.53 seconds
Tokens per second: 188.79
Draft token acceptance rate: 100.00%
  Speculative: 0.54s, 185.87 tok/s
  Baseline:    0.81s, 123.90 tok/s
  Speedup: 1.50x  |  Latency reduction: 33.34%

Benchmarking Promp

In [9]:
sweep_ks = [2, 4, 8, 16]
max_tokens = 100
num_runs = 3

sweep_rows = []

for k in sweep_ks:
    print(f"\n=== num_speculative_tokens = {k} ===")

    speedups = []
    acceptance_rates = []
    spec_tps = []
    baseline_tps = []

    for prompt in test_prompts:
        results = decoder.benchmark(
            prompt=prompt,
            max_tokens=max_tokens,
            num_runs=num_runs,
            compare_baseline=True,
            num_speculative_tokens=k,
        )

        speedups.append(results["speedup"])
        acceptance_rates.append(results["speculative"]["avg_acceptance_rate"])
        spec_tps.append(results["speculative"]["avg_tokens_per_second"])
        baseline_tps.append(results["baseline"]["avg_tokens_per_second"])

    row = {
        "k": k,
        "acceptance_rate": sum(acceptance_rates) / len(acceptance_rates),
        "speedup": sum(speedups) / len(speedups),
        "speculative_tok_s": sum(spec_tps) / len(spec_tps),
        "baseline_tok_s": sum(baseline_tps) / len(baseline_tps),
    }
    sweep_rows.append(row)

print("\n| k | acceptance rate | speedup | speculative tok/s | baseline tok/s |")
print("|---:|---:|---:|---:|---:|")
for row in sweep_rows:
    print(
        f"| {row['k']} "
        f"| {row['acceptance_rate']:.2%} "
        f"| {row['speedup']:.2f}x "
        f"| {row['speculative_tok_s']:.2f} "
        f"| {row['baseline_tok_s']:.2f} |"
    )


=== num_speculative_tokens = 2 ===
Generated 100 tokens in 0.66 seconds
Tokens per second: 152.33
Draft token acceptance rate: 97.06%
Generated 100 tokens in 0.66 seconds
Tokens per second: 151.14
Draft token acceptance rate: 97.06%
Generated 100 tokens in 0.67 seconds
Tokens per second: 149.69
Draft token acceptance rate: 97.06%
Generated 100 tokens in 0.65 seconds
Tokens per second: 154.27
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.66 seconds
Tokens per second: 151.91
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.64 seconds
Tokens per second: 155.97
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.70 seconds
Tokens per second: 143.55
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.70 seconds
Tokens per second: 143.55
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.68 seconds
Tokens per second: 146.69
Draft token acceptance rate: 100.00%

=== num_speculative_tokens = 4 ===
Generated 100 tokens in 0.62 se

In [11]:
def run_one_setting(decoder, label, k=8, max_tokens=100, num_runs=3):
    speedups = []
    acceptance_rates = []
    spec_tps = []
    baseline_tps = []

    for prompt in test_prompts:
        results = decoder.benchmark(
            prompt=prompt,
            max_tokens=max_tokens,
            num_runs=num_runs,
            compare_baseline=True,
            num_speculative_tokens=k,
        )
        speedups.append(results["speedup"])
        acceptance_rates.append(results["speculative"]["avg_acceptance_rate"])
        spec_tps.append(results["speculative"]["avg_tokens_per_second"])
        baseline_tps.append(results["baseline"]["avg_tokens_per_second"])

    row = {
        "label": label,
        "k": k,
        "acceptance_rate": sum(acceptance_rates) / len(acceptance_rates),
        "speedup": sum(speedups) / len(speedups),
        "speculative_tok_s": sum(spec_tps) / len(spec_tps),
        "baseline_tok_s": sum(baseline_tps) / len(baseline_tps),
    }

    print(
        f"{label}: k={k}, "
        f"accept={row['acceptance_rate']:.2%}, "
        f"speedup={row['speedup']:.2f}x, "
        f"spec={row['speculative_tok_s']:.2f} tok/s, "
        f"base={row['baseline_tok_s']:.2f} tok/s"
    )
    return row


def repeat_ablation(decoder, label, use_cache, repeats=3, k=8, max_tokens=100, num_runs=3):
    rows = []
    decoder.draft_use_cache = use_cache

    for r in range(repeats):
        print(f"\n--- {label}, repeat {r + 1}/{repeats} ---")
        row = run_one_setting(
            decoder,
            label=f"{label} repeat {r + 1}",
            k=k,
            max_tokens=max_tokens,
            num_runs=num_runs,
        )
        rows.append(row)

    avg = {
        "label": label,
        "k": k,
        "acceptance_rate": sum(row["acceptance_rate"] for row in rows) / repeats,
        "speedup": sum(row["speedup"] for row in rows) / repeats,
        "speculative_tok_s": sum(row["speculative_tok_s"] for row in rows) / repeats,
        "baseline_tok_s": sum(row["baseline_tok_s"] for row in rows) / repeats,
    }

    print(
        f"\nAVERAGE {label}: k={k}, "
        f"accept={avg['acceptance_rate']:.2%}, "
        f"speedup={avg['speedup']:.2f}x, "
        f"spec={avg['speculative_tok_s']:.2f} tok/s, "
        f"base={avg['baseline_tok_s']:.2f} tok/s"
    )
    return avg, rows


cache_avg, cache_rows = repeat_ablation(
    decoder,
    label="fp16 + draft KV cache",
    use_cache=True,
    repeats=3,
    k=8,
)

no_cache_avg, no_cache_rows = repeat_ablation(
    decoder,
    label="fp16 + no draft KV cache",
    use_cache=False,
    repeats=3,
    k=8,
)

decoder.draft_use_cache = True


--- fp16 + draft KV cache, repeat 1/3 ---
Generated 100 tokens in 0.88 seconds
Tokens per second: 113.98
Draft token acceptance rate: 91.67%
Generated 100 tokens in 0.59 seconds
Tokens per second: 170.40
Draft token acceptance rate: 91.67%
Generated 100 tokens in 0.58 seconds
Tokens per second: 171.10
Draft token acceptance rate: 91.67%
Generated 100 tokens in 0.54 seconds
Tokens per second: 185.34
Draft token acceptance rate: 100.00%
Generated 100 tokens in 2.96 seconds
Tokens per second: 33.83
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.53 seconds
Tokens per second: 187.74
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.57 seconds
Tokens per second: 174.84
Draft token acceptance rate: 93.68%
Generated 100 tokens in 0.58 seconds
Tokens per second: 171.14
Draft token acceptance rate: 93.68%
Generated 100 tokens in 0.56 seconds
Tokens per second: 178.73
Draft token acceptance rate: 93.68%
fp16 + draft KV cache repeat 1: k=8, accept=95.12%, speedup=1.09

## Bonus 3.B — Tree speculation or n-gram lookup decoding (10 pts)

Implement one stronger speculative-decoding variant and benchmark it
against the baseline:

- **Tree / multi-branch speculation** (Medusa / EAGLE-2 style): verify
  several candidate continuations in a single target forward pass.
- **N-gram lookup decoding** (Prompt Lookup Decoding): draft the next
  tokens from an n-gram cache built over the running sequence instead of
  (or in addition to) the draft model.

Re-run the benchmark with your bonus decoder and report the speedup and
acceptance rate in your write-up. See the bonus rubric in `../README.md`.

In [12]:
# Bonus implementation goes here.
# Re-run the benchmark above with your bonus decoder and copy the numbers into
# your report.

class NGramLookupSpeculativeDecoder(SpeculativeDecoder):
    """Speculative decoder that tries n-gram lookup before the draft model."""

    def _lookup_ngram_tokens(
        self,
        input_ids: torch.Tensor,
        num_speculative_tokens: int,
    ) -> Tuple[Optional[torch.Tensor], int]:
        ids = input_ids[0].tolist()
        min_ngram = getattr(self, "min_ngram", 2)
        max_ngram = min(getattr(self, "max_ngram", 6), len(ids) // 2)

        for n in range(max_ngram, min_ngram - 1, -1):
            suffix_start = len(ids) - n
            suffix = ids[suffix_start:]

            # Search most recent non-overlapping occurrence of the suffix.
            for start in range(suffix_start - n, -1, -1):
                if ids[start:start + n] != suffix:
                    continue

                cont_start = start + n
                cont_end = min(cont_start + num_speculative_tokens, len(ids))
                continuation = ids[cont_start:cont_end]

                if continuation:
                    tokens = torch.tensor(
                        [continuation],
                        dtype=input_ids.dtype,
                        device=input_ids.device,
                    )
                    return tokens, n

        return None, 0

    def generate_draft_tokens(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        num_speculative_tokens: int = 10,
    ) -> torch.Tensor:
        self.ngram_calls += 1
        ngram_tokens, match_order = self._lookup_ngram_tokens(
            input_ids,
            num_speculative_tokens,
        )

        if ngram_tokens is not None:
            self.ngram_hits += 1
            self.ngram_tokens_proposed += ngram_tokens.shape[1]
            self.ngram_match_orders.append(match_order)
            return ngram_tokens

        self.draft_fallbacks += 1
        return super().generate_draft_tokens(
            input_ids,
            attention_mask,
            num_speculative_tokens=num_speculative_tokens,
        )

    def speculative_decode(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_speculative_tokens: int = 8,
    ) -> str:
        self.ngram_calls = 0
        self.ngram_hits = 0
        self.ngram_tokens_proposed = 0
        self.ngram_match_orders = []
        self.draft_fallbacks = 0

        text = super().speculative_decode(
            prompt,
            max_tokens=max_tokens,
            num_speculative_tokens=num_speculative_tokens,
        )

        self.last_stats.update({
            "ngram_calls": self.ngram_calls,
            "ngram_hits": self.ngram_hits,
            "ngram_hit_rate": self.ngram_hits / self.ngram_calls if self.ngram_calls else 0,
            "ngram_tokens_proposed": self.ngram_tokens_proposed,
            "draft_fallbacks": self.draft_fallbacks,
            "avg_ngram_match_order": (
                sum(self.ngram_match_orders) / len(self.ngram_match_orders)
                if self.ngram_match_orders else 0
            ),
        })
        return text


def make_ngram_lookup_decoder(base_decoder, min_ngram=2, max_ngram=6):
    """Reuse the already-loaded target/draft models without loading another copy."""
    bonus_decoder = NGramLookupSpeculativeDecoder.__new__(NGramLookupSpeculativeDecoder)
    bonus_decoder.__dict__ = base_decoder.__dict__.copy()
    bonus_decoder.min_ngram = min_ngram
    bonus_decoder.max_ngram = max_ngram
    return bonus_decoder


def run_bonus_setting(decoder, label, prompts, k=8, max_tokens=100, num_runs=3):
    speedups = []
    acceptance_rates = []
    spec_tps = []
    baseline_tps = []
    ngram_hit_rates = []

    for prompt in prompts:
        results = decoder.benchmark(
            prompt=prompt,
            max_tokens=max_tokens,
            num_runs=num_runs,
            compare_baseline=True,
            num_speculative_tokens=k,
        )

        speedups.append(results["speedup"])
        acceptance_rates.append(results["speculative"]["avg_acceptance_rate"])
        spec_tps.append(results["speculative"]["avg_tokens_per_second"])
        baseline_tps.append(results["baseline"]["avg_tokens_per_second"])
        ngram_hit_rates.append(decoder.last_stats.get("ngram_hit_rate", 0))

    row = {
        "label": label,
        "k": k,
        "acceptance_rate": sum(acceptance_rates) / len(acceptance_rates),
        "speedup": sum(speedups) / len(speedups),
        "speculative_tok_s": sum(spec_tps) / len(spec_tps),
        "baseline_tok_s": sum(baseline_tps) / len(baseline_tps),
        "ngram_hit_rate": sum(ngram_hit_rates) / len(ngram_hit_rates),
    }

    print(
        f"{label}: k={k}, "
        f"accept={row['acceptance_rate']:.2%}, "
        f"speedup={row['speedup']:.2f}x, "
        f"spec={row['speculative_tok_s']:.2f} tok/s, "
        f"base={row['baseline_tok_s']:.2f} tok/s, "
        f"ngram_hit={row['ngram_hit_rate']:.2%}"
    )
    return row

In [ ]:
bonus_prompts = [
    "Happy birthday to you, happy birthday to you, happy birthday dear friend, happy birthday to",
    "The pattern is red blue green, red blue green, red blue green, red blue",
    "def add_numbers(a, b):\n    return a + b\n\ndef multiply_numbers(a, b):\n    return a",
]

ngram_decoder = make_ngram_lookup_decoder(decoder, min_ngram=2, max_ngram=6)

ngram_bonus_row = run_bonus_setting(
    ngram_decoder,
    "n-gram lookup + draft fallback",
    prompts=test_prompts,
    k=8,
    max_tokens=100,
    num_runs=3,
)

Generated 100 tokens in 0.41 seconds
Tokens per second: 242.99
Draft token acceptance rate: 80.56%
Generated 100 tokens in 0.22 seconds
Tokens per second: 463.82
Draft token acceptance rate: 80.56%
Generated 100 tokens in 0.20 seconds
Tokens per second: 490.66
Draft token acceptance rate: 80.56%
Generated 100 tokens in 0.11 seconds
Tokens per second: 933.13
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.12 seconds
Tokens per second: 811.07
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.11 seconds
Tokens per second: 880.80
Draft token acceptance rate: 100.00%
Generated 100 tokens in 0.58 seconds
Tokens per second: 172.60
Draft token acceptance rate: 56.55%
Generated 100 tokens in 0.55 seconds
Tokens per second: 182.13
Draft token acceptance rate: 56.55%
Generated 100 tokens in 0.56 seconds
Tokens per second: 178.01
Draft token acceptance rate: 56.55%
n-gram lookup + draft fallback: k=8, accept=79.04%, speedup=3.87x, spec=483.91 tok/s, base=120.53 tok/s, n